[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JanNikolaidovic/python-text-analytics-lab/blob/main/labs/lab10_text_as_vectors/lab10_student.ipynb)

# Lab 10 — Text as Vectors

### Digital Skills and Business Decisioning Using Python

So far you've treated text as text: strings to clean, split, and count. But almost every method in data analysis, from a simple average to a machine learning model, works on **numbers**. To analyse text at scale, you first have to turn it into numbers in a way that keeps some of its meaning.

This lab shows the most fundamental way to do that: represent every review as a **vector** (a list of numbers, one per word). Once reviews are vectors, you can *measure* things about them that were fuzzy before, like "how similar are these two reviews?". That one question is behind search engines ("find reviews like this one"), recommendations, duplicate detection, and much of the text analytics you'll do in the follow-up course.

You'll build everything with plain Python lists and functions first, so that nothing is hidden. In the next lab, NumPy will do the same work in a fraction of the code. In this lab you'll learn to:

* describe what a **vector** is, and why its positions matter,
* turn a set of reviews into **bag-of-words** vectors over a shared vocabulary,
* compute the **dot product** of two vectors (and use `zip()` to walk two lists at once),
* compute a vector's **norm** (its length),
* combine the two into **cosine similarity**, a score from 0 to 1 for how alike two texts are, and
* use it to find the most similar pair of reviews, and to **search** reviews with a query.

### Part 1: What Is a Vector?

A **vector** is an ordered list of numbers, where **each position has a fixed meaning**. You've used vectors already without the name: a product described as `[price, average_rating]` is a vector with two positions, or **dimensions**.

Suppose we describe three reviews by counting just two words, `love` and `return`:

| Review | `love` | `return` | Vector |
|---|---|---|---|
| A: *"love it, love the color, love the fit"* | 3 | 0 | `[3, 0]` |
| B: *"had to return it, return label included"* | 0 | 2 | `[0, 2]` |
| C: *"love the style but will return it"* | 1 | 1 | `[1, 1]` |

The order is the whole point: `[3, 0]` means "3 × love, 0 × return" **only** because we agreed that position 0 is `love` and position 1 is `return`. Swap the order for one review and the numbers become meaningless.

With two dimensions, each vector is a point (or an arrow from the origin) on a flat plane. Run the cell below to draw them. You don't need to understand the plotting code yet: charts are the topic of Lab 15.

In [ ]:
import matplotlib.pyplot as plt

vectors = {"A": [3, 0], "B": [0, 2], "C": [1, 1]}

fig, ax = plt.subplots(figsize=(4, 4))
for name, (x, y) in vectors.items():
    ax.annotate("", xy=(x, y), xytext=(0, 0), arrowprops={"arrowstyle": "->", "lw": 2})
    ax.text(x + 0.1, y + 0.1, name, fontsize=12)
ax.set_xlim(0, 3.5)
ax.set_ylim(0, 3.5)
ax.set_xlabel("count of 'love'")
ax.set_ylabel("count of 'return'")
ax.set_title("Three reviews as 2-D vectors")
plt.show()

The picture already suggests something useful. A points along the `love` axis and B along the `return` axis: they're about completely different things, and their arrows are at a right angle. C sits in between. **The angle between two arrows is a measure of how similar the two texts are**, and by the end of this lab you'll be able to compute it.

Real reviews use thousands of different words, so real review vectors have thousands of dimensions. Nobody can draw that, but the good news is that the maths is identical in 2 dimensions or 3,000. Build your intuition on the picture; let the code handle the rest.

---

### Part 2: Bag of Words, Turning Reviews Into Vectors

To turn a collection of reviews into vectors, you need to agree on the positions first. The standard recipe is called **bag of words**:

1. **Tokenize** every review (Labs 08 and 09).
2. Build the **vocabulary**: every distinct word across *all* the reviews, in a fixed (say, alphabetical) order.
3. For each review, count how many times each vocabulary word appears. That list of counts is the review's vector.

Here are four short reviews to work with, and a regex tokenizer from Lab 09 (letters and apostrophes only):

In [ ]:
import re

docs = [
    "Love the color, love the fit",
    "The fit is great and the color is great",
    "Had to return it, the fit was wrong",
    "Return label included, easy return",
]

def tokenize(text):
    return re.findall(r"[a-z']+", text.lower())

tokenized = [tokenize(doc) for doc in docs]
for tokens in tokenized:
    print(tokens)

Step 2, the vocabulary: put every token from every review into one set (duplicates vanish), then sort it so the positions are fixed and predictable.

In [ ]:
all_words = set()
for tokens in tokenized:
    all_words.update(tokens)

vocabulary = sorted(all_words)
print(vocabulary)
print(f"{len(vocabulary)} words, so every vector will have {len(vocabulary)} dimensions.")

(`.update()` on a set adds every item from a list, just like `Counter.update()` did in Lab 08.)

Step 3: for each review, count every vocabulary word. A list's `.count(x)` method returns how many times `x` appears in the list, so a list comprehension does the whole job:

In [ ]:
def vectorize(tokens, vocabulary):
    return [tokens.count(word) for word in vocabulary]

vectors = [vectorize(tokens, vocabulary) for tokens in tokenized]

for vector in vectors:
    print(vector)

Four reviews, four lists of 16 numbers, and every list uses the same positions. Stacked like this, they form a **document-term matrix**: one row per document (review), one column per term (word). It's easier to read with the words as column headers:

In [ ]:
print(" " * 5 + "".join(f"{word[:7]:>8}" for word in vocabulary))
for i, vector in enumerate(vectors):
    print(f"doc{i}" + " " + "".join(f"{count:>8}" for count in vector))

(Two small new tools here. `enumerate(vectors)` hands you each item **together with its position**, as `i, vector`, which saves you from keeping a separate counter. And `word[:7]` shortens long words so the columns fit.)

Most of each row is zeros: any single review uses only a few words from the vocabulary. With a realistic vocabulary of 3,000 words, the rows are more than 99% zeros. That's normal, and it's the reason special tools (like the NumPy and scikit-learn libraries) exist to handle these matrices efficiently.

### Why "Bag" of Words?

The vectors record *which* words appear and *how often*, but not **in what order**. It's as if you cut the review into words and shook them in a bag. That's a big simplification, and it has consequences:

In [ ]:
a = tokenize("not good, bad")
b = tokenize("good, not bad")

small_vocabulary = sorted(set(a) | set(b))
print(small_vocabulary)
print(vectorize(a, small_vocabulary))
print(vectorize(b, small_vocabulary))

Two opposite opinions, identical vectors. Bag of words is surprisingly effective anyway (most reviews are long enough that word choice alone says a lot), but remember its blind spot. More advanced representations that keep word order are a topic for the follow-up course.

### Practice: Bag of Words

**Exercise 1: Build a Vocabulary (easy)**

Tokenize the two reviews below with `tokenize()`, build their shared vocabulary as a sorted list, and print it together with its length.

*Expected result:* `['and', 'but', 'color', 'cute', 'runs', 'small', 'soft']` and `7`

In [ ]:
review_1 = "Soft and cute"
review_2 = "Cute color but runs small"

# TODO: tokenize both, build the sorted shared vocabulary, print it and its length

In [ ]:
# @solution
review_1 = "Soft and cute"
review_2 = "Cute color but runs small"

vocab = sorted(set(tokenize(review_1)) | set(tokenize(review_2)))
print(vocab)
print(len(vocab))

**Exercise 2: Vectorize a New Review (medium)**

Using the 16-word `vocabulary` from above, vectorize the new review below and print the vector. Then answer in a comment: the review contains the word `perfect`, so why isn't it counted anywhere in the vector?

In [ ]:
new_review = "Perfect fit, love the color and the fit"

# TODO: vectorize new_review over vocabulary and print it; explain in a comment what happened to "perfect"

In [ ]:
# @solution
new_review = "Perfect fit, love the color and the fit"

print(vectorize(tokenize(new_review), vocabulary))
# "perfect" is not in the vocabulary (it never appeared in the four original docs),
# so the vector has no position for it and the word is simply ignored.

---

### Part 3: The Dot Product

The **dot product** of two vectors of the same length multiplies them position by position and adds up the results:

$$a \cdot b = a_0 b_0 + a_1 b_1 + \dots + a_{n-1} b_{n-1}$$

For `a = [3, 0, 1]` and `b = [2, 5, 1]`: `3×2 + 0×5 + 1×1 = 7`.

To compute it in Python you need to walk two lists **side by side**. The built-in **`zip()`** does exactly that: it pairs up the first items, the second items, and so on.

In [ ]:
a = [3, 0, 1]
b = [2, 5, 1]

for x, y in zip(a, b):
    print(x, y, x * y)

Each pass of the loop gets one pair, unpacked into `x` and `y`. Add up the products with the accumulator pattern, and you have the dot product:

In [ ]:
def dot(a, b):
    total = 0
    for x, y in zip(a, b):
        total += x * y
    return total

print(dot([3, 0, 1], [2, 5, 1]))

### What Does the Dot Product Mean for Text?

In a word-count vector, a product `x * y` is **non-zero only when both reviews use that word**. So the dot product adds up the words two reviews share, weighted by how often each uses them. Words used by only one of the two reviews contribute nothing.

Compare every pair of our four reviews:

In [ ]:
for i in range(len(docs)):
    for j in range(i + 1, len(docs)):
        print(f"doc{i} · doc{j} = {dot(vectors[i], vectors[j])}")

(The inner loop starts at `i + 1` so that each pair is compared once, and no review is compared with itself.)

`doc0` and `doc1` score highest: both talk about `the`, `color`, and `fit`. `doc0` and `doc3` score `0`: they share no words at all. So far, so sensible.

But there's a problem. Watch what happens when one review is simply **longer**:

In [ ]:
short = [1, 1, 0]
long = [4, 4, 0]
other = [1, 1, 1]

print(dot(short, other))
print(dot(long, other))

`long` is just `short` written four times over (the same words, in the same proportions), yet its dot product with `other` is four times bigger. The dot product mixes up two things: **what** a review talks about and **how long** it is. To compare meaning fairly, you have to take length out of the picture. That's what the norm is for.

### Practice: Dot Product

**Exercise 3: Dot Product by Hand (easy)**

Work out the dot product of `p` and `q` on paper first, then check it with `dot()`.

*Expected result:* `12`

In [ ]:
p = [2, 1, 0, 3]
q = [1, 4, 5, 2]

# TODO: print dot(p, q)

In [ ]:
# @solution
p = [2, 1, 0, 3]
q = [1, 4, 5, 2]

print(dot(p, q))  # 2*1 + 1*4 + 0*5 + 3*2 = 12

**Exercise 4: A One-Line Dot Product (medium)**

Rewrite `dot()` as `dot_short(a, b)` whose body is a single `return` line, using `sum()` with a comprehension over `zip(a, b)` (like the `sum(... for ...)` from Lab 08). Check that it gives the same result as `dot()` for `p` and `q`.

In [ ]:
# TODO: define dot_short(a, b) in one return line, then print dot_short(p, q) and dot(p, q)

In [ ]:
# @solution
def dot_short(a, b):
    return sum(x * y for x, y in zip(a, b))

print(dot_short(p, q), dot(p, q))

---

### Part 4: The Norm, a Vector's Length

The **norm** of a vector is its length: how far the arrow reaches from the origin. In 2 dimensions it's Pythagoras' theorem: the vector `[3, 4]` has length $\sqrt{3^2 + 4^2} = \sqrt{25} = 5$.

The same formula works in any number of dimensions: square every entry, add them up, take the square root.

$$\|a\| = \sqrt{a_0^2 + a_1^2 + \dots + a_{n-1}^2}$$

Notice that "square every entry and add them up" is just the dot product of a vector **with itself**, so the norm can reuse `dot()`:

In [ ]:
import math

def norm(a):
    return math.sqrt(dot(a, a))

print(norm([3, 4]))
print(norm([1, 1, 0]))
print(norm([4, 4, 0]))

`math.sqrt()` is the square root from the `math` module (Lab 06). The long review's vector is exactly four times as long as the short one's, which is precisely the length effect we want to remove.

### Unit Vectors

Divide every entry of a vector by its norm, and you get a vector pointing in the **same direction** with a length of exactly `1`: a **unit vector**. Scaling to unit length throws away "how long" and keeps only "which direction", in other words, the *mix* of words, regardless of how many there are.

In [ ]:
def unit(a):
    length = norm(a)
    return [x / length for x in a]

print(unit([1, 1, 0]))
print(unit([4, 4, 0]))
print(norm(unit([4, 4, 0])))

After scaling, `short` and `long` become the same vector. (The last line may print `0.9999999999999999` instead of `1.0`. Computers store decimals with tiny rounding errors, which is harmless here.)

### Practice: Norm

**Exercise 5: Norm of a Review (easy)**

Print the norm of `vectors[1]` (the second review, `"The fit is great and the color is great"`), rounded to 2 decimal places.

*Expected result:* `3.87`

In [ ]:
# TODO: print the norm of vectors[1], rounded to 2 decimals

In [ ]:
# @solution
print(round(norm(vectors[1]), 2))

---

### Part 5: Cosine Similarity

Now put the pieces together. The dot product of two **unit** vectors measures how much they point the same way, with length removed. There's a neat shortcut: instead of scaling both vectors first, divide the dot product by both norms. The result is the **cosine** of the angle between the two arrows, which is why it's called **cosine similarity**:

$$\text{cosine}(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|}$$

For word counts (which are never negative), cosine similarity always lands between **0** and **1**:

* **1**: the two reviews use words in exactly the same proportions (the arrows point the same way),
* **0**: they share no words at all (the arrows are at a right angle),
* anything in between: partial overlap.

In [ ]:
def cosine(a, b):
    return dot(a, b) / (norm(a) * norm(b))

print(cosine([1, 1, 0], [4, 4, 0]))
print(cosine([3, 0], [0, 2]))
print(cosine([3, 0], [1, 1]))

* `short` and `long` score `1` (printed as `0.9999999999999998`, the same harmless rounding as before): the same words in the same proportions, so their length no longer matters.
* Reviews A and B from Part 1 score `0.0`: nothing in common, arrows at a right angle.
* A and C score about `0.71`, the cosine of the 45° angle you saw in the picture.

### Guarding Against Empty Reviews

A review with no words (or none from the vocabulary) becomes a vector of all zeros, whose norm is `0`, and dividing by zero raises the `ZeroDivisionError` from Lab 07. A safe version of `cosine()` checks first:

In [ ]:
def cosine(a, b):
    denominator = norm(a) * norm(b)
    if denominator == 0:
        return 0.0
    return dot(a, b) / denominator

print(cosine([0, 0, 0], [1, 2, 3]))

Returning `0.0` means "no similarity", which is a sensible answer for an empty review. Now compare the four example reviews again, this time with cosine similarity:

In [ ]:
for i in range(len(docs)):
    for j in range(i + 1, len(docs)):
        print(f"doc{i} vs doc{j}: {cosine(vectors[i], vectors[j]):.2f}")

Every score is now on the same 0-to-1 scale, whatever the length of the reviews, so scores can be compared directly and a threshold (say, "above 0.8 means probably a duplicate") means the same thing for every pair.

### Practice: Cosine Similarity

**Exercise 6: Which Pair Is Closer? (easy)**

Compute the cosine similarity of `x` with `y` and of `x` with `z`, each rounded to 3 decimals. Which vector points in a direction closer to `x`?

*Expected result:* `0.984` and `0.632`

In [ ]:
x = [2, 1, 0]
y = [4, 3, 0]
z = [1, 0, 1]

# TODO: print cosine(x, y) and cosine(x, z), rounded to 3 decimals

In [ ]:
# @solution
x = [2, 1, 0]
y = [4, 3, 0]
z = [1, 0, 1]

print(round(cosine(x, y), 3))
print(round(cosine(x, z), 3))
# y is much closer to x in direction, even though it is "longer".

---

### Part 6: Similarity on Real Reviews

Here are eight real reviews from the dataset (a few from each star rating). Which two are the most alike? Let's find out with the tools you just built.

In [ ]:
reviews = [
    "Gorgeous top but runs large, especially in the arm holes.",
    "This is an attractive shirt but runs big. i ordered the black and white.",
    "This dress is really really soft and comfortable. it's loose but still flattering. i would definitely recommend it.",
    "Love this sweater! great for work. very soft and comfortable. color is gorgeous!",
    "This shirt is unflattering and boxy. it does not look cute at all on me. i am returning.",
    "I'm returning this cardigan. it doesn't fit me at all. it's shapeless and loose.",
    "Great casual weekend pants and super comfy and great color.",
    "The print on these leggings is so cool but they are completely see-through. i mean seriously, seriously see-through.",
]

def build_vectors(texts, stopwords=set()):
    token_lists = [[t for t in tokenize(text) if t not in stopwords] for text in texts]
    vocab = sorted(set(t for tokens in token_lists for t in tokens))
    return vocab, [vectorize(tokens, vocab) for tokens in token_lists]

vocab, review_vectors = build_vectors(reviews)
print(f"Vocabulary: {len(vocab)} words")

`build_vectors()` packages Part 2 into one function: tokenize every text, drop any stopwords you pass in, build the vocabulary, and vectorize. (The nested comprehension `t for tokens in token_lists for t in tokens` reads like two `for` loops written left to right: for each list of tokens, for each token in it.)

Now compare every pair and keep the best one, with the "best so far" pattern from Lab 05:

In [ ]:
def most_similar_pair(vectors):
    best_score = -1
    best_pair = None
    for i in range(len(vectors)):
        for j in range(i + 1, len(vectors)):
            score = cosine(vectors[i], vectors[j])
            if score > best_score:
                best_score = score
                best_pair = (i, j)
    return best_pair, best_score

(i, j), score = most_similar_pair(review_vectors)
print(f"Most similar ({score:.2f}):")
print("  ", reviews[i])
print("  ", reviews[j])

Both reviews are about returning an item, so the answer looks sensible. But look at *which* words they actually share: `returning`, and then `this`, `is`, `and`, `it`, `at`, `all`. Most of the score comes from small function words that almost every review contains. Remove them with a stopword set, as in Lab 08, and compare:

In [ ]:
STOPWORDS = {"a", "an", "the", "and", "but", "is", "it", "it's", "this", "i", "i'm", "me",
             "on", "in", "for", "so", "at", "all", "they", "these", "am", "are", "very", "would"}

vocab_sw, review_vectors_sw = build_vectors(reviews, STOPWORDS)
(i, j), score = most_similar_pair(review_vectors_sw)
print(f"Vocabulary: {len(vocab_sw)} words")
print(f"Most similar without stopwords ({score:.2f}):")
print("  ", reviews[i])
print("  ", reviews[j])

A different pair wins. With the filler words gone, the two "returning" reviews share only one word, while these two enthusiastic reviews share `great` and `color`. Neither answer is "the" right one: they answer slightly different questions, and which one you get depends on a cleaning choice you made. That's the same lesson as Lab 08, seen from a new angle: **the choices you make when cleaning text directly change the numbers your analysis produces.**

### Searching With a Query

Cosine similarity also gives you a tiny search engine. Treat a search query as one more (very short) document, vectorize it over the **same** vocabulary, and rank every review by its similarity to the query.

In [ ]:
def search(query, texts, vocab, vectors, stopwords=set()):
    query_tokens = [t for t in tokenize(query) if t not in stopwords]
    query_vector = vectorize(query_tokens, vocab)
    scored = [(cosine(query_vector, vector), text) for vector, text in zip(vectors, texts)]
    return sorted(scored, reverse=True)

for score, text in search("soft comfortable", reviews, vocab_sw, review_vectors_sw, STOPWORDS)[:3]:
    print(f"{score:.2f}  {text}")

`sorted(scored, reverse=True)` sorts the `(score, text)` tuples by their **first** element (the score), highest first, which is the tuple-sorting rule from Lab 08. The two "soft and comfortable" reviews come out on top, and everything that doesn't share a word with the query scores `0`.

This is the core of how document search worked for decades. Modern systems replace word counts with smarter vectors, but "turn text into vectors, rank by cosine similarity" is still how it's done.

### Practice: Real Reviews

**Exercise 7: Search for Fit Problems (easy)**

Use `search()` to find the 3 reviews most similar to the query `"runs big large"` (with `STOPWORDS`), and print each score (2 decimals) and text.

In [ ]:
# TODO: search for "runs big large" and print the top 3 results

In [ ]:
# @solution
for score, text in search("runs big large", reviews, vocab_sw, review_vectors_sw, STOPWORDS)[:3]:
    print(f"{score:.2f}  {text}")

**Exercise 8: The Least Similar Review (medium)**

Write a function `average_similarity(index, vectors)` that returns the **average** cosine similarity between review number `index` and every *other* review. Then loop over all eight reviews (with stopwords removed, `review_vectors_sw`) and print which review is the **least** similar to the rest on average: the "odd one out".

In [ ]:
# TODO: define average_similarity(index, vectors), then find and print the review with the lowest average

In [ ]:
# @solution
def average_similarity(index, vectors):
    scores = []
    for j in range(len(vectors)):
        if j != index:
            scores.append(cosine(vectors[index], vectors[j]))
    return sum(scores) / len(scores)

lowest_index = 0
lowest_score = average_similarity(0, review_vectors_sw)
for i in range(1, len(review_vectors_sw)):
    score = average_similarity(i, review_vectors_sw)
    if score < lowest_score:
        lowest_index = i
        lowest_score = score

print(f"Odd one out ({lowest_score:.3f}): {reviews[lowest_index]}")

---

### Recap: Text as Vectors

| Idea / Tool | What It Means | Example |
|---|---|---|
| Vector | An ordered list of numbers, each position with a fixed meaning | `[3, 0]` = 3 × love, 0 × return |
| Vocabulary | The sorted list of every distinct word; fixes the positions | `sorted(set(all_tokens))` |
| Bag of words | A review's word counts over the vocabulary (order is lost) | `[tokens.count(w) for w in vocab]` |
| Document-term matrix | One row per document, one column per word | `[vectorize(t, vocab) for t in tokenized]` |
| `zip(a, b)` | Walk two lists side by side, in pairs | `for x, y in zip(a, b):` |
| `enumerate(list)` | Loop with each item's position | `for i, v in enumerate(vectors):` |
| Dot product | Sum of position-by-position products; rewards shared words | `sum(x * y for x, y in zip(a, b))` |
| Norm | A vector's length: `sqrt` of the sum of squares | `math.sqrt(dot(a, a))` |
| Unit vector | Same direction, length 1 | `[x / norm(a) for x in a]` |
| Cosine similarity | `dot / (norm × norm)`: 0 = nothing shared, 1 = same word mix | `cosine(a, b)` |

The whole lab in one sentence: **a bag-of-words vector counts each vocabulary word in a text, and cosine similarity (the dot product divided by both norms) compares two texts by their mix of words, regardless of their length.**